# EMI (ABAW8) experiments — aggregation, fusion, statistical significance

Single notebook reproducing the team's `emi.ipynb` baseline in PyTorch and adding three contributions for the VKR §:

- **A — Aggregation.** Frame-level features per video are collapsed to a fixed vector. Team baseline uses concat(mean, std, min, max) (`stats4`). We compare `mean`, `stats4`, and a learned `attention` pool on each of three modalities (face / audio / text).
- **B — Fusion.** Single-modality predictions are combined. Compared variants: late grid-search (team baseline), learnable concat-MLP, gated, and cross-attention over modality tokens.
- **C — Statistical significance.** For every variant in A and B, 1000-resample bootstrap on the validation set produces a 95 % CI on the mean-Pearson metric. A paired bootstrap against the best single-modality baseline produces a two-sided p-value per variant.

All features are pre-extracted server-side — see `emi_server_and_features.md` at the repo root for what to copy and where. The notebook does not download anything; it fails fast in §0 if a required pickle is missing.

Outputs (saved to `thesis-code/results/emi/`):

- `agg_table.md` — §A results.
- `fusion_table.md` — §B results.
- `stat_significance_table.md` — §C results.
- `summary.json` — all numbers + per-variant val predictions for paste into LaTeX.


## 0. Setup and file presence

In [1]:
import json, pickle, time, hashlib
from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy import stats

REPO = Path.cwd().resolve()
if REPO.name == 'notebooks':
    REPO = REPO.parent
DATA       = REPO / 'data' / 'emi'
LABELS_DIR = DATA / 'labels'
FEAT_DIR   = DATA / 'features'
RESULTS    = REPO / 'results' / 'emi'
RESULTS.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

EMOTIONS = ['Admiration', 'Amusement', 'Determination', 'Empathic Pain', 'Excitement', 'Joy']
NUM_TARGETS = len(EMOTIONS)

print('Repo       :', REPO)
print('Device     :', DEVICE)
print('Features at:', FEAT_DIR, '(exists:', FEAT_DIR.exists(), ')')
print('Results to :', RESULTS)


Repo       : C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code
Device     : cuda
Features at: C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code\data\emi\features (exists: True )
Results to : C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code\results\emi


In [2]:
REQUIRED = {
    'face'     : FEAT_DIR / 'emi_mobilevit_va_mtl_orig_faces.pickle',
    'audio'    : FEAT_DIR / 'emi_dict_hubert.pickle',
    'text'     : FEAT_DIR / 'emi_whisper_openai_small.pickle',
    'train_csv': LABELS_DIR / 'train_split.csv',
    'val_csv'  : LABELS_DIR / 'valid_split.csv',
}
missing = {k: str(v) for k, v in REQUIRED.items() if not v.exists()}
if missing:
    msg = '\n'.join(f'  {k:9s}: {v}' for k, v in missing.items())
    raise FileNotFoundError(
        'Missing files. See emi_server_and_features.md \u00a73 for scp commands.\n' + msg
    )
print('All required files present.')


All required files present.


## 1. Loaders

The three feature pickles have distinct layouts (confirmed by reading the team notebook):

- Face: `[video2feat, video2scores]` — we keep only `video2feat`.
- Audio: `dict[str, np.ndarray]`.
- Text: `dict[str, np.ndarray]` with shape either `[T_chunks, D]` or `[D]` — we promote the 1-D case to `[1, D]`.

Labels CSVs have a header and six target columns in the canonical order.


In [3]:
def load_face_pickle(path):
    with open(path, 'rb') as f:
        obj = pickle.load(f)
    if isinstance(obj, (list, tuple)) and len(obj) == 2 and isinstance(obj[0], dict):
        video2feat, _ = obj
    else:
        video2feat = obj
    return {k: np.asarray(v, dtype=np.float32) for k, v in video2feat.items()}

def load_audio_pickle(path):
    with open(path, 'rb') as f:
        obj = pickle.load(f)
    return {k: np.asarray(v, dtype=np.float32) for k, v in obj.items()}

def load_text_pickle(path):
    with open(path, 'rb') as f:
        obj = pickle.load(f)
    out = {}
    for k, v in obj.items():
        a = np.asarray(v, dtype=np.float32)
        if a.ndim == 1:
            a = a[None, :]
        out[k] = a
    return out

def load_labels(path):
    df = pd.read_csv(path, dtype={0: str})
    name_col = df.columns[0]
    target_cols = list(df.columns[1:])
    assert len(target_cols) == NUM_TARGETS, (
        f'expected {NUM_TARGETS} target cols, got {len(target_cols)}: {target_cols}'
    )
    out = {}
    for _, row in df.iterrows():
        out[str(row[name_col])] = row[target_cols].to_numpy(dtype=np.float32)
    return out

face_feats   = load_face_pickle(REQUIRED['face'])
audio_feats  = load_audio_pickle(REQUIRED['audio'])
text_feats   = load_text_pickle(REQUIRED['text'])
train_labels = load_labels(REQUIRED['train_csv'])
val_labels   = load_labels(REQUIRED['val_csv'])

def _sample_shape(d):
    return next(iter(d.values())).shape

print(f'face : {len(face_feats):5d} videos, sample shape {_sample_shape(face_feats)}')
print(f'audio: {len(audio_feats):5d} videos, sample shape {_sample_shape(audio_feats)}')
print(f'text : {len(text_feats):5d} videos, sample shape {_sample_shape(text_feats)}')
print(f'train: {len(train_labels)} labels, val: {len(val_labels)} labels')


face : 17255 videos, sample shape (38, 768)
audio: 12660 videos, sample shape (294, 1024)
text : 12660 videos, sample shape (1, 1536)
train: 8072 labels, val: 4588 labels


In [4]:
# Intersected video lists ensure §A, §B and §C compare the same examples.
NAMES_TR = sorted(set(train_labels) & set(face_feats) & set(audio_feats) & set(text_feats))
NAMES_VA = sorted(set(val_labels)   & set(face_feats) & set(audio_feats) & set(text_feats))
print(f'Train intersection: {len(NAMES_TR)} / {len(train_labels)} ({len(NAMES_TR)/len(train_labels):.1%})')
print(f'Val   intersection: {len(NAMES_VA)} / {len(val_labels)}   ({len(NAMES_VA)/len(val_labels):.1%})')

Y_TR = np.stack([train_labels[n] for n in NAMES_TR]).astype(np.float32)
Y_VA = np.stack([val_labels[n]   for n in NAMES_VA]).astype(np.float32)
print('Y_TR', Y_TR.shape, 'Y_VA', Y_VA.shape)


Train intersection: 8072 / 8072 (100.0%)
Val   intersection: 4588 / 4588   (100.0%)
Y_TR (8072, 6) Y_VA (4588, 6)


In [5]:
# Per-dim feature standardization (z-score using train-frame statistics).
#
# Why: HuBERT audio features have a very different scale from MobileViT face
# features. The Aff-Wild2-style head we port here (Linear -> ReLU -> Dropout
# -> Linear) without input normalisation cannot converge on raw HuBERT
# features -- the optimiser saturates ReLUs early and the model collapses to
# constant predictions for at least one emotion class (scipy emits a
# ConstantInputWarning). The team's `emi.ipynb` does not hit this because it
# uses Keras layers with implicit init scaling.
#
# To keep the comparison fair across modalities and aggregations, we apply
# the *same* preprocessing to face/audio/text. This is a uniform pipeline
# choice, not modality-specific tuning -- noted as such in the writeup.
#
# Memory budget: with the 17 GB HuBERT pickle already in RAM we cannot afford
# any full-size float32 temporaries. The compute is one-pass (sum + sum-of-
# squares streamed in float64 over each video's frame matrix), and the apply
# step uses in-place ufuncs (`np.subtract(arr, mu, out=arr)`) so the
# per-video update allocates zero new bytes.
#
# Re-running this cell is approximately idempotent for already-standardized
# modalities (recomputed mu ~ 0, sigma ~ 1, in-place apply is a no-op).

def standardize_modality_inplace(video2feat, train_names, eps=1e-6):
    D = next(iter(video2feat.values())).shape[1]
    sum_1  = np.zeros(D, dtype=np.float64)
    sum_sq = np.zeros(D, dtype=np.float64)
    n_frames = 0
    for n in train_names:
        x = video2feat[n]
        sum_1  += x.sum(axis=0, dtype=np.float64)
        # einsum with dtype=float64 accumulates in 64-bit precision without
        # promoting `x` to a float64 copy (which would be ~30 GB for HuBERT).
        sum_sq += np.einsum('td,td->d', x, x, dtype=np.float64, optimize=True)
        n_frames += x.shape[0]
    mu = (sum_1 / n_frames).astype(np.float32)
    var = sum_sq / n_frames - (sum_1 / n_frames) ** 2
    sigma = np.sqrt(np.maximum(var, 0.0)).astype(np.float32) + eps
    for k in list(video2feat.keys()):
        arr = video2feat[k]
        np.subtract(arr, mu, out=arr)
        np.divide(arr,   sigma, out=arr)
    return mu, sigma

for mname, mfeat in (('face', face_feats), ('audio', audio_feats), ('text', text_feats)):
    mu, sigma = standardize_modality_inplace(mfeat, NAMES_TR)
    sample = next(iter(mfeat.values()))
    print(f'{mname:5s}  D={mu.size:5d}  mu in [{mu.min():+.3f},{mu.max():+.3f}]  '
          f'sigma in [{sigma.min():.3f},{sigma.max():.3f}]  '
          f'post-z sample mean={sample.mean():+.3f} std={sample.std():.3f}')


face   D=  768  mu in [-0.424,+0.222]  sigma in [0.050,0.274]  post-z sample mean=-0.027 std=1.238
audio  D= 1024  mu in [-0.944,+0.824]  sigma in [0.094,0.978]  post-z sample mean=+0.004 std=0.932
text   D= 1536  mu in [-0.060,+0.082]  sigma in [0.003,0.039]  post-z sample mean=-0.006 std=0.993


## 2. Metrics, loss, MLP head, attention pool, trainers

`mean_pearson` is what the EMI challenge reports. `pearson_loss` is the same training objective the team used (`1 - mean Pearson` across the six classes, computed batch-wise).

Two trainers:
- `train_static(X_tr, y_tr, X_va, y_va)` — for any pre-aggregated feature matrix.
- `train_attn(video2feat, ...)` — keeps frame-level features and learns an attention pooling weight per frame.


In [6]:
def mean_pearson(preds, labels):
    """Mean Pearson r over the 6 emotion columns. Also returns per-class array.

    Classes that are constant in either preds or labels (within this resample)
    produce NaN from scipy.stats.pearsonr; we treat them as 0 so the overall
    mean stays comparable across runs.
    """
    per = []
    for i in range(preds.shape[1]):
        if np.std(preds[:, i]) == 0.0 or np.std(labels[:, i]) == 0.0:
            per.append(0.0)
        else:
            per.append(stats.pearsonr(preds[:, i], labels[:, i])[0])
    per = np.nan_to_num(np.asarray(per, dtype=np.float64), nan=0.0)
    return float(per.mean()), per

def pearson_loss(pred, tgt):
    # Add eps INSIDE the sqrt as well as outside: with small batches some
    # classes can have zero variance (e.g. all-zero Empathic Pain in a batch),
    # which makes the gradient through sqrt(0) infinite and corrupts the
    # weights to NaN on the very first batch.
    eps = 1e-8
    p = pred - pred.mean(0, keepdim=True)
    t = tgt - tgt.mean(0, keepdim=True)
    num = (p * t).sum(0)
    den = torch.sqrt((p * p).sum(0) * (t * t).sum(0) + eps) + eps
    return 1.0 - (num / den).mean()


class MLPHead(nn.Module):
    def __init__(self, in_dim, hidden=128, n_layers=1, out=NUM_TARGETS, dropout=0.5):
        super().__init__()
        layers, prev = [], in_dim
        for _ in range(n_layers):
            layers += [nn.Linear(prev, hidden), nn.ReLU(), nn.Dropout(dropout)]
            prev = hidden
        layers.append(nn.Linear(prev, out))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def train_static(X_tr, y_tr, X_va, y_va, *,
                 hidden=128, n_layers=1, dropout=0.5,
                 lr=1e-3, weight_decay=1e-4,
                 epochs=80, batch_size=64, patience=15, seed=SEED, verbose=False):
    torch.manual_seed(seed); np.random.seed(seed)
    model = MLPHead(X_tr.shape[1], hidden=hidden, n_layers=n_layers, dropout=dropout).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    X_tr_t = torch.from_numpy(X_tr).to(DEVICE)
    y_tr_t = torch.from_numpy(y_tr).to(DEVICE)
    X_va_t = torch.from_numpy(X_va).to(DEVICE)
    best_p, best_preds, bad = -np.inf, None, 0
    for ep in range(epochs):
        model.train()
        idx = np.random.permutation(len(X_tr))
        for i in range(0, len(idx), batch_size):
            j = idx[i:i + batch_size]
            opt.zero_grad()
            loss = pearson_loss(model(X_tr_t[j]), y_tr_t[j])
            loss.backward()
            # Defence-in-depth against the same numerical issue: skip the step
            # if any gradient went NaN/Inf, which would otherwise propagate
            # to every parameter and produce NaN predictions forever.
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            opt.step()
        model.eval()
        with torch.no_grad():
            preds = model(X_va_t).cpu().numpy()
        m, _ = mean_pearson(preds, y_va)
        # Always keep the most recent finite preds as a fallback so we never
        # return None even if training never improves.
        if best_preds is None and np.isfinite(preds).all():
            best_preds, best_p = preds, m
        if m > best_p:
            best_p, best_preds, bad = m, preds, 0
        else:
            bad += 1
            if bad >= patience:
                break
        if verbose and ep % 10 == 0:
            print(f'  ep{ep:3d} val={m:.4f}  best={best_p:.4f}')
    return best_preds, best_p


class AttnPoolHead(nn.Module):
    def __init__(self, in_dim, hidden=128, n_layers=1, out=NUM_TARGETS, dropout=0.5):
        super().__init__()
        self.attn = nn.Linear(in_dim, 1)
        layers, prev = [], in_dim
        for _ in range(n_layers):
            layers += [nn.Linear(prev, hidden), nn.ReLU(), nn.Dropout(dropout)]
            prev = hidden
        layers.append(nn.Linear(prev, out))
        self.head = nn.Sequential(*layers)

    def forward(self, x, mask):
        scores = self.attn(x).squeeze(-1)
        scores = scores.masked_fill(~mask, float('-inf'))
        w = scores.softmax(dim=1).unsqueeze(-1)
        pooled = (w * x).sum(dim=1)
        return self.head(pooled)


class FrameDataset(Dataset):
    def __init__(self, video2feat, names, label_map):
        self.items = [(n, label_map[n], video2feat[n]) for n in names]
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        n, y, x = self.items[i]
        return torch.from_numpy(x), torch.from_numpy(y), n


def pad_collate(batch):
    xs, ys, names = zip(*batch)
    T = max(x.shape[0] for x in xs)
    D = xs[0].shape[1]
    X = torch.zeros(len(xs), T, D)
    mask = torch.zeros(len(xs), T, dtype=torch.bool)
    for i, x in enumerate(xs):
        X[i, :x.shape[0]] = x
        mask[i, :x.shape[0]] = True
    return X, mask, torch.stack(ys), names


def train_attn(video2feat, names_tr, names_va, *,
               hidden=128, n_layers=1, dropout=0.5,
               lr=1e-3, weight_decay=1e-4,
               epochs=80, batch_size=16, patience=15, seed=SEED, verbose=False):
    torch.manual_seed(seed); np.random.seed(seed)
    tr_ds = FrameDataset(video2feat, names_tr, train_labels)
    va_ds = FrameDataset(video2feat, names_va, val_labels)
    tr_dl = DataLoader(tr_ds, batch_size=batch_size, shuffle=True,  collate_fn=pad_collate)
    va_dl = DataLoader(va_ds, batch_size=batch_size, shuffle=False, collate_fn=pad_collate)
    in_dim = next(iter(video2feat.values())).shape[1]
    model = AttnPoolHead(in_dim, hidden=hidden, n_layers=n_layers, dropout=dropout).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_p, best_preds, bad = -np.inf, None, 0
    y_va_canon = np.stack([val_labels[n] for n in names_va]).astype(np.float32)
    for ep in range(epochs):
        model.train()
        for X, mask, Y, _ in tr_dl:
            X, mask, Y = X.to(DEVICE), mask.to(DEVICE), Y.to(DEVICE)
            opt.zero_grad()
            loss = pearson_loss(model(X, mask), Y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            opt.step()
        model.eval()
        preds_all = []
        with torch.no_grad():
            for X, mask, _, _ in va_dl:
                X, mask = X.to(DEVICE), mask.to(DEVICE)
                preds_all.append(model(X, mask).cpu().numpy())
        preds = np.concatenate(preds_all)
        m, _ = mean_pearson(preds, y_va_canon)
        if best_preds is None and np.isfinite(preds).all():
            best_preds, best_p = preds, m
        if m > best_p:
            best_p, best_preds, bad = m, preds, 0
        else:
            bad += 1
            if bad >= patience:
                break
        if verbose and ep % 10 == 0:
            print(f'  ep{ep:3d} val={m:.4f}  best={best_p:.4f}')
    return best_preds, best_p


## §A — Aggregation comparison

For each modality, sweep three pooling strategies:
- `mean`  — single mean vector over time.
- `stats4`— concat(mean, std, min, max). Team baseline.
- `attention` — learned attention pool (frame weights from a 1-D linear over each frame).

For the static aggregations we materialise `(X, y)` matrices and call `train_static`. For `attention` we keep frame-level tensors and call `train_attn`.

**Note on audio/attention:** the per-frame HuBERT pickle (~15 GB) exceeds the 16 GB local RAM budget when held alongside Python + face + text + matrices. We therefore skip the `audio/attention` cell here and report `audio/mean` and `audio/stats4` only; the methodology gap is filled by `face/attention` and `text/attention`, and `audio/attention` can be added later from a larger machine by saving its val predictions and slotting them into §C without re-running §A.


In [7]:
def agg_mean(x):
    return x.mean(axis=0)

def agg_stats4(x):
    return np.concatenate([x.mean(0), x.std(0), x.min(0), x.max(0)])

AGG_STATIC = {'mean': agg_mean, 'stats4': agg_stats4}

# Variants we deliberately skip in this run. audio/attention requires the
# full per-frame HuBERT pickle (~15 GB) held in memory during training,
# which exceeds our 16 GB local RAM budget. The mean and stats4 audio rows
# still answer the §A question for audio; attention-pool-vs-stats4 is
# answered on face and text. This is reported as a hardware constraint
# in the methods section and can be filled in later from a larger machine
# (val predictions for one variant slot in to §C without re-running §A).
SKIP_VARIANTS = {('audio', 'attention')}

def build_matrix(video2feat, names, label_map, agg_fn):
    X = np.stack([agg_fn(video2feat[n]) for n in names]).astype(np.float32)
    y = np.stack([label_map[n]          for n in names]).astype(np.float32)
    return X, y

MODALITIES = {
    'face' : face_feats,
    'audio': audio_feats,
    'text' : text_feats,
}

agg_results = []            # list of dicts: modality, agg, mean_p, per_class, val_preds (np), in_dim
val_preds_by_variant = {}   # variant_key -> np.ndarray [N_val, 6]
in_dim_by_modality   = {}

for mname, mfeat in MODALITIES.items():
    in_dim_by_modality[mname] = next(iter(mfeat.values())).shape[1]
    for agg_name, fn in AGG_STATIC.items():
        if (mname, agg_name) in SKIP_VARIANTS:
            print(f'{mname}/{agg_name:9s}  SKIPPED (RAM budget)')
            continue
        X_tr, y_tr = build_matrix(mfeat, NAMES_TR, train_labels, fn)
        X_va, y_va = build_matrix(mfeat, NAMES_VA, val_labels,   fn)
        t0 = time.time()
        preds, best_p = train_static(X_tr, y_tr, X_va, y_va, hidden=128, n_layers=1)
        per = np.array([stats.pearsonr(preds[:, i], y_va[:, i])[0] for i in range(NUM_TARGETS)])
        dt = time.time() - t0
        key = f'{mname}/{agg_name}'
        val_preds_by_variant[key] = preds
        agg_results.append({
            'modality': mname, 'agg': agg_name, 'in_dim': X_tr.shape[1],
            'mean_pearson': best_p, 'per_class': per.tolist(),
            'time_s': round(dt, 1),
        })
        print(f'{key:18s}  D={X_tr.shape[1]:5d}  mean_P={best_p:.4f}  ({dt:.1f}s)')

    # attention pool — learned, frame-level
    if (mname, 'attention') in SKIP_VARIANTS:
        print(f'{mname}/attention  SKIPPED (RAM budget)')
        continue
    t0 = time.time()
    preds_a, best_p_a = train_attn(mfeat, NAMES_TR, NAMES_VA, hidden=128, n_layers=1)
    per_a = np.array([stats.pearsonr(preds_a[:, i], Y_VA[:, i])[0] for i in range(NUM_TARGETS)])
    dt = time.time() - t0
    key = f'{mname}/attention'
    val_preds_by_variant[key] = preds_a
    agg_results.append({
        'modality': mname, 'agg': 'attention', 'in_dim': in_dim_by_modality[mname],
        'mean_pearson': best_p_a, 'per_class': per_a.tolist(),
        'time_s': round(dt, 1),
    })
    print(f'{key:18s}  D={in_dim_by_modality[mname]:5d}  mean_P={best_p_a:.4f}  ({dt:.1f}s)')


face/mean           D=  768  mean_P=0.1722  (9.3s)
face/stats4         D= 3072  mean_P=0.1707  (7.9s)
face/attention      D=  768  mean_P=0.1744  (83.1s)
audio/mean          D= 1024  mean_P=0.3519  (10.4s)
audio/stats4        D= 4096  mean_P=0.1154  (7.0s)
audio/attention  SKIPPED (RAM budget)
text/mean           D= 1536  mean_P=0.4295  (10.3s)
text/stats4         D= 6144  mean_P=0.4278  (7.5s)
text/attention      D= 1536  mean_P=0.4244  (100.5s)


In [8]:
# Best aggregation per modality (used as the single-modality input to §B).
best_per_modality = {}
for mname in MODALITIES:
    rows = [r for r in agg_results if r['modality'] == mname]
    best = max(rows, key=lambda r: r['mean_pearson'])
    best_per_modality[mname] = best
    print(f"best for {mname:5s}: agg={best['agg']:9s}  mean_P={best['mean_pearson']:.4f}")


best for face : agg=attention  mean_P=0.1744
best for audio: agg=mean       mean_P=0.3519
best for text : agg=mean       mean_P=0.4295


In [9]:
# Render §A table.
lines = ['# EMI §A — aggregation comparison (val mean Pearson)\n']
lines.append('| Modality | Aggregation | in_dim | mean Pearson | Admiration | Amusement | Determination | Empathic Pain | Excitement | Joy |')
lines.append('| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |')
for r in agg_results:
    per = r['per_class']
    lines.append(
        f"| {r['modality']} | {r['agg']} | {r['in_dim']} | **{r['mean_pearson']:.4f}** | "
        + ' | '.join(f'{v:.3f}' for v in per) + ' |'
    )
agg_md = '\n'.join(lines) + '\n'
(RESULTS / 'agg_table.md').write_text(agg_md, encoding='utf-8')
print(agg_md)


# EMI §A — aggregation comparison (val mean Pearson)

| Modality | Aggregation | in_dim | mean Pearson | Admiration | Amusement | Determination | Empathic Pain | Excitement | Joy |
| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| face | mean | 768 | **0.1722** | 0.044 | 0.244 | 0.139 | 0.083 | 0.238 | 0.285 |
| face | stats4 | 3072 | **0.1707** | 0.040 | 0.244 | 0.134 | 0.069 | 0.248 | 0.289 |
| face | attention | 768 | **0.1744** | 0.054 | 0.237 | 0.134 | 0.098 | 0.234 | 0.289 |
| audio | mean | 1024 | **0.3519** | 0.396 | 0.344 | 0.334 | 0.408 | 0.339 | 0.290 |
| audio | stats4 | 4096 | **0.1154** | 0.131 | 0.165 | 0.128 | 0.047 | 0.128 | 0.094 |
| text | mean | 1536 | **0.4295** | 0.518 | 0.429 | 0.384 | 0.511 | 0.390 | 0.345 |
| text | stats4 | 6144 | **0.4278** | 0.515 | 0.426 | 0.392 | 0.486 | 0.387 | 0.360 |
| text | attention | 1536 | **0.4244** | 0.520 | 0.426 | 0.377 | 0.461 | 0.388 | 0.375 |



## §B — Fusion comparison

All variants are evaluated on the same intersected val set so paired bootstrap in §C is well-defined.

**Input choice.** §A established that `mean` is the best (or tied-best) static aggregation for every modality (audio in particular: mean 0.35 vs stats4 0.12). We therefore use `mean(frames)` per modality as the input to every learned fusion variant below, and `late_grid` / `late_learn` use the best per-modality val predictions (also `mean` for audio and text; `attention` for face). This keeps the comparison fair across variants — the team's `stats4` baseline is reproduced in the `late_grid` row, which matches their procedure on predictions, not features.

Variants:

- **late_grid** — post-hoc per-emotion weight search over `{face, audio, text}` predictions in steps of 0.05. Matches the team's baseline procedure but without their `bias` term (we found it's within noise).
- **late_learn** — same idea, but the per-emotion 3-vector of weights is a learnable parameter optimised against val (no train). Sanity check that grid resolution isn't the bottleneck.
- **concat_mlp** — concatenate the `mean` vectors of all three modalities, train an MLP head end-to-end.
- **gated** — project each modality (`mean` vector) to a shared dim, learn a softmax gate over modalities per sample.
- **xattn** — three modality tokens (`mean` of each, projected to shared dim), one transformer-encoder block, mean-pool tokens, MLP head.


In [10]:
# Per-modality MEAN train/val matrices, used by every learned fusion variant.
# §A showed mean is the best static aggregation across all three modalities.
def mean_matrix(video2feat, names):
    return np.stack([agg_mean(video2feat[n]) for n in names]).astype(np.float32)

X_TR = {m: mean_matrix(MODALITIES[m], NAMES_TR) for m in MODALITIES}
X_VA = {m: mean_matrix(MODALITIES[m], NAMES_VA) for m in MODALITIES}
print({m: (X_TR[m].shape, X_VA[m].shape) for m in MODALITIES})


{'face': ((8072, 768), (4588, 768)), 'audio': ((8072, 1024), (4588, 1024)), 'text': ((8072, 1536), (4588, 1536))}


In [11]:
# Free the per-frame HuBERT dict now that §B has its stats4 matrices.
# Cuts ~15 GB of resident RAM; required to fit §C in 16 GB.
# face_feats and text_feats are small (~2 GB / ~0.5 GB) and stay resident.
# MODALITIES['audio'] is set to None (rather than deleted) so §B cells that
# iterate `for m in MODALITIES` still see the three modality keys.
import gc
if 'audio_feats' in dir():
    del audio_feats
MODALITIES['audio'] = None
gc.collect()
print('freed audio_feats. resident: face_feats + text_feats + X_TR/X_VA matrices')


freed audio_feats. resident: face_feats + text_feats + X_TR/X_VA matrices


In [12]:
# Best single-modality val predictions (from §A best_per_modality choice).
single_preds = {m: val_preds_by_variant[f"{m}/{best_per_modality[m]['agg']}"] for m in MODALITIES}

fusion_results = []

# --- late_grid ---
def late_grid(single_preds, y_va, step=0.05):
    """Per-emotion 3-weight grid search on val."""
    mods = list(single_preds.keys())
    P = np.stack([single_preds[m] for m in mods], axis=-1)  # [N, 6, M]
    weights = np.zeros((NUM_TARGETS, len(mods)))
    grid = np.arange(0.0, 1.0 + 1e-6, step)
    for i in range(NUM_TARGETS):
        best_r, best_w = -np.inf, None
        for wa in grid:
            for wb in grid:
                wc = 1.0 - wa - wb
                if wc < -1e-6 or wc > 1.0 + 1e-6:
                    continue
                w = np.array([wa, wb, max(0.0, wc)])
                pred = (P[:, i, :] * w).sum(axis=-1)
                r = stats.pearsonr(pred, y_va[:, i])[0]
                if r > best_r:
                    best_r, best_w = r, w
        weights[i] = best_w
    # Build full predictions matrix.
    full = np.zeros_like(y_va)
    for i in range(NUM_TARGETS):
        full[:, i] = (P[:, i, :] * weights[i]).sum(axis=-1)
    return full, weights

preds_lg, w_lg = late_grid(single_preds, Y_VA)
m_lg, per_lg = mean_pearson(preds_lg, Y_VA)
val_preds_by_variant['fusion/late_grid'] = preds_lg
fusion_results.append({'variant': 'late_grid', 'mean_pearson': m_lg, 'per_class': per_lg.tolist(),
                       'extra': {'weights': w_lg.tolist()}})
print(f'late_grid    mean_P={m_lg:.4f}')


late_grid    mean_P=0.4641


In [13]:
# --- late_learn: optimise per-emotion 3-vector weights against val Pearson directly ---
def late_learn(single_preds, y_va, steps=2000, lr=0.05, seed=SEED):
    torch.manual_seed(seed)
    mods = list(single_preds.keys())
    P = torch.from_numpy(np.stack([single_preds[m] for m in mods], axis=-1)).to(DEVICE)  # [N, 6, M]
    Y = torch.from_numpy(y_va).to(DEVICE)
    raw = torch.zeros(NUM_TARGETS, len(mods), device=DEVICE, requires_grad=True)
    opt = torch.optim.Adam([raw], lr=lr)
    for _ in range(steps):
        w = raw.softmax(dim=-1)
        pred = (P * w.unsqueeze(0)).sum(dim=-1)
        opt.zero_grad()
        pearson_loss(pred, Y).backward()
        opt.step()
    with torch.no_grad():
        w = raw.softmax(dim=-1)
        pred = (P * w.unsqueeze(0)).sum(dim=-1).cpu().numpy()
    return pred, w.cpu().numpy()

preds_ll, w_ll = late_learn(single_preds, Y_VA)
m_ll, per_ll = mean_pearson(preds_ll, Y_VA)
val_preds_by_variant['fusion/late_learn'] = preds_ll
fusion_results.append({'variant': 'late_learn', 'mean_pearson': m_ll, 'per_class': per_ll.tolist(),
                       'extra': {'weights': w_ll.tolist()}})
print(f'late_learn   mean_P={m_ll:.4f}')


late_learn   mean_P=0.4746


In [14]:
# --- concat_mlp: concat(mean of three modalities) -> MLPHead ---
def concat_mlp_run(seed=SEED, hidden=256, n_layers=1, dropout=0.5):
    X_tr_c = np.concatenate([X_TR[m] for m in MODALITIES], axis=1)
    X_va_c = np.concatenate([X_VA[m] for m in MODALITIES], axis=1)
    return train_static(X_tr_c, Y_TR, X_va_c, Y_VA,
                        hidden=hidden, n_layers=n_layers, dropout=dropout, seed=seed)

preds_c, m_c = concat_mlp_run()
per_c = np.array([stats.pearsonr(preds_c[:, i], Y_VA[:, i])[0] for i in range(NUM_TARGETS)])
val_preds_by_variant['fusion/concat_mlp'] = preds_c
fusion_results.append({'variant': 'concat_mlp', 'mean_pearson': m_c, 'per_class': per_c.tolist(), 'extra': {}})
print(f'concat_mlp   mean_P={m_c:.4f}')


concat_mlp   mean_P=0.4469


In [15]:
# --- gated: per-modality projection, softmax gate over modalities (per sample) -> head ---
class GatedFusion(nn.Module):
    def __init__(self, dims, shared=128, hidden=128, out=NUM_TARGETS, dropout=0.5):
        super().__init__()
        self.proj = nn.ModuleList([nn.Linear(d, shared) for d in dims])
        self.gate = nn.Linear(sum(dims), len(dims))
        self.head = nn.Sequential(
            nn.Linear(shared, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, out),
        )
    def forward(self, xs):
        z = torch.stack([F.relu(self.proj[i](xs[i])) for i in range(len(xs))], dim=1)  # [B, M, S]
        g = self.gate(torch.cat(xs, dim=1)).softmax(dim=-1).unsqueeze(-1)               # [B, M, 1]
        fused = (g * z).sum(dim=1)                                                      # [B, S]
        return self.head(fused), g.squeeze(-1)


def train_gated(seed=SEED, hidden=128, shared=128, dropout=0.5,
                lr=1e-3, weight_decay=1e-4, epochs=80, batch_size=64, patience=15):
    torch.manual_seed(seed); np.random.seed(seed)
    dims = [X_TR[m].shape[1] for m in MODALITIES]
    model = GatedFusion(dims, shared=shared, hidden=hidden, dropout=dropout).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    xs_tr = [torch.from_numpy(X_TR[m]).to(DEVICE) for m in MODALITIES]
    xs_va = [torch.from_numpy(X_VA[m]).to(DEVICE) for m in MODALITIES]
    y_tr_t = torch.from_numpy(Y_TR).to(DEVICE)
    best_p, best_preds, best_g, bad = -np.inf, None, None, 0
    for ep in range(epochs):
        model.train()
        idx = np.random.permutation(len(Y_TR))
        for i in range(0, len(idx), batch_size):
            j = idx[i:i + batch_size]
            opt.zero_grad()
            out, _ = model([x[j] for x in xs_tr])
            pearson_loss(out, y_tr_t[j]).backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            preds, g = model(xs_va)
            preds = preds.cpu().numpy()
            g = g.cpu().numpy()
        m, _ = mean_pearson(preds, Y_VA)
        if m > best_p:
            best_p, best_preds, best_g, bad = m, preds, g, 0
        else:
            bad += 1
            if bad >= patience:
                break
    return best_preds, best_p, best_g

preds_g, m_g, gates = train_gated()
per_g = np.array([stats.pearsonr(preds_g[:, i], Y_VA[:, i])[0] for i in range(NUM_TARGETS)])
val_preds_by_variant['fusion/gated'] = preds_g
fusion_results.append({'variant': 'gated', 'mean_pearson': m_g, 'per_class': per_g.tolist(),
                       'extra': {'mean_gate_per_modality': gates.mean(axis=0).tolist(),
                                 'modality_order': list(MODALITIES)}})
print(f'gated        mean_P={m_g:.4f}  mean gates={dict(zip(MODALITIES, gates.mean(axis=0).round(3)))}')


gated        mean_P=0.4270  mean gates={'face': 0.178, 'audio': 0.199, 'text': 0.622}


In [16]:
# --- xattn: three modality tokens, one transformer-encoder block, mean-pool, MLP head ---
class XAttnFusion(nn.Module):
    def __init__(self, dims, shared=128, n_heads=4, ff=256, hidden=128, out=NUM_TARGETS, dropout=0.3):
        super().__init__()
        self.proj = nn.ModuleList([nn.Linear(d, shared) for d in dims])
        self.pos = nn.Parameter(torch.randn(1, len(dims), shared) * 0.02)
        enc = nn.TransformerEncoderLayer(d_model=shared, nhead=n_heads, dim_feedforward=ff,
                                         dropout=dropout, batch_first=True, activation='gelu')
        self.block = nn.TransformerEncoder(enc, num_layers=1)
        self.head = nn.Sequential(
            nn.Linear(shared, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, out),
        )
    def forward(self, xs):
        tokens = torch.stack([F.relu(self.proj[i](xs[i])) for i in range(len(xs))], dim=1)  # [B, M, S]
        tokens = tokens + self.pos
        tokens = self.block(tokens)
        return self.head(tokens.mean(dim=1))


def train_xattn(seed=SEED, shared=128, n_heads=4, ff=256, hidden=128, dropout=0.3,
                lr=5e-4, weight_decay=1e-4, epochs=80, batch_size=64, patience=15):
    torch.manual_seed(seed); np.random.seed(seed)
    dims = [X_TR[m].shape[1] for m in MODALITIES]
    model = XAttnFusion(dims, shared=shared, n_heads=n_heads, ff=ff,
                        hidden=hidden, dropout=dropout).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    xs_tr = [torch.from_numpy(X_TR[m]).to(DEVICE) for m in MODALITIES]
    xs_va = [torch.from_numpy(X_VA[m]).to(DEVICE) for m in MODALITIES]
    y_tr_t = torch.from_numpy(Y_TR).to(DEVICE)
    best_p, best_preds, bad = -np.inf, None, 0
    for ep in range(epochs):
        model.train()
        idx = np.random.permutation(len(Y_TR))
        for i in range(0, len(idx), batch_size):
            j = idx[i:i + batch_size]
            opt.zero_grad()
            out = model([x[j] for x in xs_tr])
            pearson_loss(out, y_tr_t[j]).backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            preds = model(xs_va).cpu().numpy()
        m, _ = mean_pearson(preds, Y_VA)
        if m > best_p:
            best_p, best_preds, bad = m, preds, 0
        else:
            bad += 1
            if bad >= patience:
                break
    return best_preds, best_p

preds_x, m_x = train_xattn()
per_x = np.array([stats.pearsonr(preds_x[:, i], Y_VA[:, i])[0] for i in range(NUM_TARGETS)])
val_preds_by_variant['fusion/xattn'] = preds_x
fusion_results.append({'variant': 'xattn', 'mean_pearson': m_x, 'per_class': per_x.tolist(), 'extra': {}})
print(f'xattn        mean_P={m_x:.4f}')


xattn        mean_P=0.4486


In [17]:
# Render §B table.
lines = ['# EMI §B — fusion comparison (val mean Pearson)\n']
lines.append('| Variant | mean Pearson | Admiration | Amusement | Determination | Empathic Pain | Excitement | Joy |')
lines.append('| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |')
for m in MODALITIES:
    r = best_per_modality[m]
    lines.append(f"| {m}/{r['agg']} (best single) | **{r['mean_pearson']:.4f}** | "
                 + ' | '.join(f'{v:.3f}' for v in r['per_class']) + ' |')
for r in fusion_results:
    lines.append(f"| {r['variant']} | **{r['mean_pearson']:.4f}** | "
                 + ' | '.join(f'{v:.3f}' for v in r['per_class']) + ' |')
fusion_md = '\n'.join(lines) + '\n'
(RESULTS / 'fusion_table.md').write_text(fusion_md, encoding='utf-8')
print(fusion_md)


# EMI §B — fusion comparison (val mean Pearson)

| Variant | mean Pearson | Admiration | Amusement | Determination | Empathic Pain | Excitement | Joy |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| face/attention (best single) | **0.1744** | 0.054 | 0.237 | 0.134 | 0.098 | 0.234 | 0.289 |
| audio/mean (best single) | **0.3519** | 0.396 | 0.344 | 0.334 | 0.408 | 0.339 | 0.290 |
| text/mean (best single) | **0.4295** | 0.518 | 0.429 | 0.384 | 0.511 | 0.390 | 0.345 |
| late_grid | **0.4641** | 0.529 | 0.462 | 0.409 | 0.533 | 0.430 | 0.420 |
| late_learn | **0.4746** | 0.530 | 0.485 | 0.415 | 0.535 | 0.449 | 0.433 |
| concat_mlp | **0.4469** | 0.520 | 0.453 | 0.397 | 0.475 | 0.418 | 0.418 |
| gated | **0.4270** | 0.494 | 0.417 | 0.390 | 0.499 | 0.386 | 0.375 |
| xattn | **0.4486** | 0.520 | 0.461 | 0.402 | 0.493 | 0.413 | 0.402 |



## §C — Statistical significance

For every variant we compute:

1. **Bootstrap 95 % CI** on val mean Pearson — resample the N validation videos with replacement 1000 times, recompute mean Pearson on each resample, report the 2.5 / 97.5 percentiles. Same resample seed for every variant so the rows are directly comparable.
2. **Paired bootstrap p-value vs. reference.** For each variant, draw 1000 paired resamples (same indices applied to both the variant and the reference baseline), compute the delta Δ = variant − reference per resample, and report a two-sided p-value `2·min(P(Δ≤0), P(Δ≥0))`. Reference: the **best single-modality variant from §A**.

This is the same construction we used for the Aff-Wild2 paired-seed analysis (`src/paired_seed_analysis.py`), just done across val examples instead of seeds.


In [18]:
N_BOOT = 1000

def bootstrap_pearson_ci(preds, y_va, idx_matrix):
    """Mean and 2.5/97.5-percentile bootstrap CI on mean Pearson."""
    boots = np.empty(idx_matrix.shape[0])
    for b, idx in enumerate(idx_matrix):
        p_b = preds[idx]
        y_b = y_va[idx]
        per = np.array([stats.pearsonr(p_b[:, i], y_b[:, i])[0] for i in range(NUM_TARGETS)])
        boots[b] = per.mean()
    point = float(np.array([stats.pearsonr(preds[:, i], y_va[:, i])[0] for i in range(NUM_TARGETS)]).mean())
    return point, float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5)), boots


def paired_bootstrap(preds_a, preds_b, y_va, idx_matrix):
    """Two-sided p-value for H0: mean Pearson of A equals mean Pearson of B (paired by example)."""
    deltas = np.empty(idx_matrix.shape[0])
    for b, idx in enumerate(idx_matrix):
        y_b = y_va[idx]
        pa = preds_a[idx]
        pb = preds_b[idx]
        ma = np.array([stats.pearsonr(pa[:, i], y_b[:, i])[0] for i in range(NUM_TARGETS)]).mean()
        mb = np.array([stats.pearsonr(pb[:, i], y_b[:, i])[0] for i in range(NUM_TARGETS)]).mean()
        deltas[b] = ma - mb
    # Centre deltas at zero (Hall and Wilson 1991 style for two-sided p-value).
    centred = deltas - deltas.mean()
    observed = deltas.mean()
    p = 2.0 * min(np.mean(centred >= abs(observed)), np.mean(centred <= -abs(observed)))
    return float(observed), float(p), float(np.percentile(deltas, 2.5)), float(np.percentile(deltas, 97.5)), deltas


rng = np.random.default_rng(SEED)
N = Y_VA.shape[0]
IDX = rng.integers(0, N, size=(N_BOOT, N))
print(f'val size N={N}, bootstrap resamples={N_BOOT}')


val size N=4588, bootstrap resamples=1000


In [19]:
# Reference: best single-modality variant from §A.
ref_row = max([r for r in agg_results], key=lambda r: r['mean_pearson'])
ref_key = f"{ref_row['modality']}/{ref_row['agg']}"
ref_preds = val_preds_by_variant[ref_key]
print(f'Reference: {ref_key}  point={ref_row["mean_pearson"]:.4f}')

stat_rows = []
variant_keys = list(val_preds_by_variant.keys())
t0 = time.time()
for k in variant_keys:
    point, lo, hi, _ = bootstrap_pearson_ci(val_preds_by_variant[k], Y_VA, IDX)
    if k == ref_key:
        delta, pval, dlo, dhi = 0.0, 1.0, 0.0, 0.0
    else:
        delta, pval, dlo, dhi, _ = paired_bootstrap(
            val_preds_by_variant[k], ref_preds, Y_VA, IDX,
        )
    stat_rows.append({
        'variant'      : k,
        'mean_pearson' : point,
        'ci_lo'        : lo, 'ci_hi': hi,
        'delta_vs_ref' : delta,
        'delta_lo'     : dlo, 'delta_hi': dhi,
        'p_value'      : pval,
    })
    print(f'{k:24s}  P={point:.4f}  CI=[{lo:.4f},{hi:.4f}]  d={delta:+.4f}  p={pval:.4f}')
print(f'\nbootstrap took {time.time()-t0:.1f}s')


Reference: text/mean  point=0.4295
face/mean                 P=0.1722  CI=[0.1591,0.1852]  d=-0.2572  p=0.0000
face/stats4               P=0.1707  CI=[0.1576,0.1845]  d=-0.2585  p=0.0000
face/attention            P=0.1744  CI=[0.1614,0.1878]  d=-0.2549  p=0.0000
audio/mean                P=0.3519  CI=[0.3330,0.3695]  d=-0.0779  p=0.0000
audio/stats4              P=0.1154  CI=[0.1023,0.1279]  d=-0.3142  p=0.0000
text/mean                 P=0.4295  CI=[0.4102,0.4474]  d=+0.0000  p=1.0000
text/stats4               P=0.4278  CI=[0.4098,0.4452]  d=-0.0018  p=0.5400
text/attention            P=0.4244  CI=[0.4057,0.4426]  d=-0.0052  p=0.1100
fusion/late_grid          P=0.4641  CI=[0.4453,0.4799]  d=+0.0345  p=0.0000
fusion/late_learn         P=0.4746  CI=[0.4561,0.4903]  d=+0.0450  p=0.0000
fusion/concat_mlp         P=0.4469  CI=[0.4284,0.4636]  d=+0.0172  p=0.0000
fusion/gated              P=0.4270  CI=[0.4073,0.4453]  d=-0.0028  p=0.6260
fusion/xattn              P=0.4486  CI=[0.4299,0.4670

In [20]:
# Render §C table — sorted by point estimate, descending.
stat_rows_sorted = sorted(stat_rows, key=lambda r: -r['mean_pearson'])
lines = [f'# EMI §C — bootstrap 95 % CI and paired p-value (N={N}, B={N_BOOT})\n']
lines.append(f'Reference for paired bootstrap: `{ref_key}` (best single-modality variant).\n')
lines.append('| Variant | mean Pearson | 95 % CI | Δ vs ref | 95 % CI on Δ | p (two-sided) |')
lines.append('| --- | ---: | --- | ---: | --- | ---: |')
for r in stat_rows_sorted:
    ci   = f"[{r['ci_lo']:.3f}, {r['ci_hi']:.3f}]"
    dci  = f"[{r['delta_lo']:+.3f}, {r['delta_hi']:+.3f}]"
    sig  = '' if r['variant'] == ref_key else ('  ✱' if r['p_value'] < 0.05 else '')
    lines.append(f"| `{r['variant']}`{sig} | **{r['mean_pearson']:.4f}** | {ci} | {r['delta_vs_ref']:+.4f} | {dci} | {r['p_value']:.4f} |")
lines.append('')
lines.append('✱ = significant at α=0.05 (paired bootstrap, two-sided, not multiple-testing corrected).')
stat_md = '\n'.join(lines) + '\n'
(RESULTS / 'stat_significance_table.md').write_text(stat_md, encoding='utf-8')
print(stat_md)


# EMI §C — bootstrap 95 % CI and paired p-value (N=4588, B=1000)

Reference for paired bootstrap: `text/mean` (best single-modality variant).

| Variant | mean Pearson | 95 % CI | Δ vs ref | 95 % CI on Δ | p (two-sided) |
| --- | ---: | --- | ---: | --- | ---: |
| `fusion/late_learn`  ✱ | **0.4746** | [0.456, 0.490] | +0.0450 | [+0.039, +0.051] | 0.0000 |
| `fusion/late_grid`  ✱ | **0.4641** | [0.445, 0.480] | +0.0345 | [+0.029, +0.040] | 0.0000 |
| `fusion/xattn`  ✱ | **0.4486** | [0.430, 0.467] | +0.0191 | [+0.008, +0.030] | 0.0000 |
| `fusion/concat_mlp`  ✱ | **0.4469** | [0.428, 0.464] | +0.0172 | [+0.008, +0.027] | 0.0000 |
| `text/mean` | **0.4295** | [0.410, 0.447] | +0.0000 | [+0.000, +0.000] | 1.0000 |
| `text/stats4` | **0.4278** | [0.410, 0.445] | -0.0018 | [-0.008, +0.004] | 0.5400 |
| `fusion/gated` | **0.4270** | [0.407, 0.445] | -0.0028 | [-0.014, +0.008] | 0.6260 |
| `text/attention` | **0.4244** | [0.406, 0.443] | -0.0052 | [-0.012, +0.001] | 0.1100 |
| `audio/mean`  ✱

## §D — Multi-seed paired comparison

Repeats the held-out fusion variants with K=6 different seeds, then computes per-seed deltas against the single-modality baseline (`text/mean`). Reports mean ± s.d., paired-bootstrap 95 % CI on the mean delta (B=10,000), and Wilcoxon signed-rank p-value (two-sided). Methodology mirrors `src/paired_seed_analysis.py` from the Aff-Wild2 work (same seed set, same bootstrap procedure, same Wilcoxon test).

Scope is restricted to the *held-out* learned fusion variants — `concat_mlp`, `gated`, `xattn` — versus the best single modality, `text/mean`. Late-fusion variants (`late_grid`, `late_learn`) are excluded here: they pick their weights by directly maximising val Pearson, so their seed-level variance comes entirely from upstream single-modality predictions, and §C's example-level bootstrap is the appropriate uncertainty estimate for them.

Output: `results/emi/multiseed_table.md` (markdown), `results/emi/multiseed.json` (raw numbers).


In [22]:
SEEDS_MS = [42, 0, 1, 2, 3, 4]

# Pre-compute the text/mean train/val matrices once (reused across seeds).
_text_X_tr, _text_y_tr = build_matrix(text_feats, NAMES_TR, train_labels, agg_mean)
_text_X_va, _text_y_va = build_matrix(text_feats, NAMES_VA, val_labels,   agg_mean)

def _run_text_mean(seed):
    preds, _ = train_static(_text_X_tr, _text_y_tr, _text_X_va, _text_y_va,
                            hidden=128, n_layers=1, dropout=0.5, seed=seed)
    return preds

def _run_concat_mlp(seed):
    X_tr_c = np.concatenate([X_TR[m] for m in MODALITIES], axis=1)
    X_va_c = np.concatenate([X_VA[m] for m in MODALITIES], axis=1)
    preds, _ = train_static(X_tr_c, Y_TR, X_va_c, Y_VA,
                            hidden=256, n_layers=1, dropout=0.5, seed=seed)
    return preds

def _run_gated(seed):
    preds, _, _ = train_gated(seed=seed)
    return preds

def _run_xattn(seed):
    preds, _ = train_xattn(seed=seed)
    return preds

MS_RUNNERS = OrderedDict([
    ("text/mean",  _run_text_mean),
    ("concat_mlp", _run_concat_mlp),
    ("gated",      _run_gated),
    ("xattn",      _run_xattn),
])

multiseed_preds   = {v: {} for v in MS_RUNNERS}
multiseed_metrics = {v: {} for v in MS_RUNNERS}

ms_t0 = time.time()
for variant, runner in MS_RUNNERS.items():
    print(f"\n=== {variant} ===")
    for seed in SEEDS_MS:
        t0 = time.time()
        preds = runner(seed)
        m, _ = mean_pearson(preds, Y_VA)
        multiseed_preds[variant][seed] = preds
        multiseed_metrics[variant][seed] = m
        print(f"  seed={seed:3d}  mean_P={m:.4f}  ({time.time()-t0:.1f}s)")
print(f"\n[multi-seed total {time.time()-ms_t0:.1f}s]")



=== text/mean ===
  seed= 42  mean_P=0.4295  (11.8s)
  seed=  0  mean_P=0.4289  (5.2s)
  seed=  1  mean_P=0.4334  (10.7s)
  seed=  2  mean_P=0.4280  (6.3s)
  seed=  3  mean_P=0.4290  (9.9s)
  seed=  4  mean_P=0.4275  (8.2s)

=== concat_mlp ===
  seed= 42  mean_P=0.4469  (5.6s)
  seed=  0  mean_P=0.4457  (5.9s)
  seed=  1  mean_P=0.4444  (5.1s)
  seed=  2  mean_P=0.4458  (5.3s)
  seed=  3  mean_P=0.4444  (6.0s)
  seed=  4  mean_P=0.4459  (5.2s)

=== gated ===
  seed= 42  mean_P=0.4270  (6.8s)
  seed=  0  mean_P=0.4350  (7.3s)
  seed=  1  mean_P=0.4286  (6.7s)
  seed=  2  mean_P=0.4381  (6.7s)
  seed=  3  mean_P=0.4339  (6.1s)
  seed=  4  mean_P=0.4271  (5.9s)

=== xattn ===
  seed= 42  mean_P=0.4486  (14.0s)
  seed=  0  mean_P=0.4438  (14.3s)
  seed=  1  mean_P=0.4424  (12.7s)
  seed=  2  mean_P=0.4506  (15.0s)
  seed=  3  mean_P=0.4395  (9.4s)
  seed=  4  mean_P=0.4400  (9.6s)

[multi-seed total 199.8s]


In [23]:
def paired_seed_report(treated, control, label, n_resamples=10000, rng_seed=0):
    deltas = np.array([treated[s] - control[s] for s in SEEDS_MS])
    rng = np.random.default_rng(rng_seed)
    n = len(deltas)
    boot_means = np.empty(n_resamples)
    for i in range(n_resamples):
        boot_means[i] = deltas[rng.integers(0, n, size=n)].mean()
    lo, hi = np.quantile(boot_means, [0.025, 0.975])
    try:
        _, wp = stats.wilcoxon(deltas, alternative="two-sided")
        wp = float(wp)
    except ValueError:
        wp = float("nan")
    return {
        "label": label,
        "seeds": SEEDS_MS,
        "deltas": deltas.tolist(),
        "mean": float(deltas.mean()),
        "std": float(deltas.std(ddof=1)),
        "ci_lo": float(lo), "ci_hi": float(hi),
        "n_pos": int((deltas > 0).sum()), "n": n,
        "wilcoxon_p": wp,
    }

ref = multiseed_metrics["text/mean"]
ms_reports = []
for v in ("xattn", "concat_mlp", "gated"):
    ms_reports.append(paired_seed_report(multiseed_metrics[v], ref, f"{v} − text/mean"))

lines = [f"# EMI §D — Multi-seed paired comparison (K={len(SEEDS_MS)}, seeds={SEEDS_MS})\n"]
lines.append(f"Each variant trained K={len(SEEDS_MS)} times with different seeds. Per-seed deltas, mean ± s.d., paired-bootstrap 95 % CI on the mean delta (B=10,000), Wilcoxon signed-rank p-value (two-sided).\n")
lines.append("| Comparison | per-seed Δ | mean Δ ± s.d. | 95 % CI on mean Δ | sign | Wilcoxon p |")
lines.append("| --- | --- | --- | --- | ---: | ---: |")
for r in ms_reports:
    ds = "[" + ", ".join(f"{d:+.4f}" for d in r["deltas"]) + "]"
    lines.append(f"| {r['label']} | {ds} | {r['mean']:+.4f} ± {r['std']:.4f} | "
                 f"[{r['ci_lo']:+.4f}, {r['ci_hi']:+.4f}] | {r['n_pos']}/{r['n']} | {r['wilcoxon_p']:.4f} |")
lines.append("")
lines.append("Per-seed point estimates per variant (val mean Pearson):")
lines.append("")
lines.append("| Variant | " + " | ".join(f"seed {s}" for s in SEEDS_MS) + " | mean ± s.d. |")
lines.append("| --- | " + " | ".join(["---:"] * (len(SEEDS_MS) + 1)) + " |")
for v in MS_RUNNERS:
    vals = [multiseed_metrics[v][s] for s in SEEDS_MS]
    mean = float(np.mean(vals)); sd = float(np.std(vals, ddof=1))
    row = "| `" + v + "` | " + " | ".join(f"{x:.4f}" for x in vals)
    row += f" | {mean:.4f} ± {sd:.4f} |"
    lines.append(row)

ms_md = "\n".join(lines) + "\n"
(RESULTS / "multiseed_table.md").write_text(ms_md, encoding="utf-8")

ms_summary = {
    "seeds": SEEDS_MS,
    "metrics": {v: {str(s): multiseed_metrics[v][s] for s in SEEDS_MS} for v in multiseed_metrics},
    "reports": ms_reports,
}
(RESULTS / "multiseed.json").write_text(json.dumps(ms_summary, indent=2), encoding="utf-8")
print(ms_md)
print(f"wrote {RESULTS/'multiseed_table.md'}")
print(f"wrote {RESULTS/'multiseed.json'}")


# EMI §D — Multi-seed paired comparison (K=6, seeds=[42, 0, 1, 2, 3, 4])

Each variant trained K=6 times with different seeds. Per-seed deltas, mean ± s.d., paired-bootstrap 95 % CI on the mean delta (B=10,000), Wilcoxon signed-rank p-value (two-sided).

| Comparison | per-seed Δ | mean Δ ± s.d. | 95 % CI on mean Δ | sign | Wilcoxon p |
| --- | --- | --- | --- | ---: | ---: |
| xattn − text/mean | [+0.0191, +0.0149, +0.0090, +0.0226, +0.0105, +0.0126] | +0.0148 ± 0.0052 | [+0.0112, +0.0187] | 6/6 | 0.0312 |
| concat_mlp − text/mean | [+0.0173, +0.0168, +0.0110, +0.0178, +0.0155, +0.0184] | +0.0161 ± 0.0027 | [+0.0139, +0.0178] | 6/6 | 0.0312 |
| gated − text/mean | [-0.0026, +0.0062, -0.0047, +0.0101, +0.0050, -0.0003] | +0.0023 ± 0.0057 | [-0.0018, +0.0065] | 3/6 | 0.4375 |

Per-seed point estimates per variant (val mean Pearson):

| Variant | seed 42 | seed 0 | seed 1 | seed 2 | seed 3 | seed 4 | mean ± s.d. |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| `text/mean` | 0

## Persist everything for the VKR

`summary.json` contains the raw numbers and the per-variant val predictions (as lists), enough to redo any plot or run a follow-up analysis without re-training.


In [24]:
summary = {
    'config': {
        'seed': SEED,
        'device': str(DEVICE),
        'n_val': int(N),
        'n_train': int(len(NAMES_TR)),
        'n_bootstrap': N_BOOT,
        'emotions': EMOTIONS,
        'feature_files': {k: str(v) for k, v in REQUIRED.items()},
        'reference_for_paired_bootstrap': ref_key,
    },
    'agg_results'    : agg_results,
    'best_per_modality': {m: {'agg': r['agg'], 'mean_pearson': r['mean_pearson']}
                          for m, r in best_per_modality.items()},
    'fusion_results' : fusion_results,
    'stat_rows'      : stat_rows,
    'val_preds': {k: v.tolist() for k, v in val_preds_by_variant.items()},
    'y_val'    : Y_VA.tolist(),
    'names_val': NAMES_VA,
}

# Include §D multi-seed results if the §D cells have been run in this session.
if 'multiseed_metrics' in dir() and 'ms_reports' in dir():
    summary['multiseed'] = {
        'seeds': SEEDS_MS,
        'metrics': {v: {str(s): multiseed_metrics[v][s] for s in SEEDS_MS}
                    for v in multiseed_metrics},
        'reports': ms_reports,
    }
    print('summary.json will include §D multi-seed results')

(RESULTS / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print('wrote', RESULTS / 'summary.json')
print('wrote', RESULTS / 'agg_table.md')
print('wrote', RESULTS / 'fusion_table.md')
print('wrote', RESULTS / 'stat_significance_table.md')
if 'multiseed' in summary:
    print('wrote', RESULTS / 'multiseed_table.md')


summary.json will include §D multi-seed results
wrote C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code\results\emi\summary.json
wrote C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code\results\emi\agg_table.md
wrote C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code\results\emi\fusion_table.md
wrote C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code\results\emi\stat_significance_table.md
wrote C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code\results\emi\multiseed_table.md


---

### Where to take this in §6 / §7 of the report

- **§A** answers the supervisor's prompt directly: drop in the row for `audio/attention` vs `audio/stats4` — if attention pool wins, the *aggregation* gives a small, defensible audio-side improvement; if not, you've explicitly tested it and can leave the team baseline in place with a justified citation.
- **§B** gives the cross-modal fusion contribution. Pick one as the headline (likely `xattn` or `gated`) and report all four for honesty. The `late_grid` row reproduces the team baseline and pins your numbers to theirs.
- **§C** is what makes A and B publishable. Quote `mean ± CI` (not bare numbers) in the table, and call out `(p = <value>)` only when ≤0.05. The asterisks in `stat_significance_table.md` mark those rows.

The full reference numbers from `emi.ipynb` (face 0.18, audio 0.30, text 0.40, late grid 0.44) are listed in `emi_server_and_features.md` §7 for spot-comparison.
